# Entrenamiento y evaluación en Google Colab

Flujo completo del proyecto: datos → entrenamiento de la ablación → evaluación → figuras → exportación de pesos.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.

**Persistencia.** Todo lo que no es temporal se escribe en `MyDrive/clifford_nvs`:

| Ruta en Drive | Contenido |
|---|---|
| `data/` | `.npz` preprocesados (se generan una sola vez) |
| `runs/<modelo>/seed<k>/` | `last.pt`, `best.pt`, `log.json`, `summary.json` |
| `results/` | tablas, figuras y CSV del artículo |
| `release/` | pesos publicables y muestra de demostración |

Cada run guarda `last.pt` cada 500 pasos o cada 10 minutos, lo que ocurra antes, con escritura atómica. Si Colab corta la sesión, se agota la cuota de GPU o se apaga tu PC, no se pierden más de 10 minutos de entrenamiento: basta con volver a ejecutar las celdas 1, 2 y 4. Los runs terminados se saltan y el interrumpido continúa desde su checkpoint con la misma secuencia aleatoria.

In [ ]:
# 1. Entorno, Drive y código
import os, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "Sin GPU: revisa el tipo de entorno")

from google.colab import drive
drive.mount("/content/drive")

WORK = "/content/drive/MyDrive/clifford_nvs"      # carpeta persistente (datos, runs, resultados)
CODE_DRIVE = "/content/drive/MyDrive/clifford-nvs"   # código subido a Drive
REPO_URL = ""   # opcional: "https://github.com/RonaldPort/clifford-nvs.git" para clonar desde GitHub en lugar de Drive
REPO = "/content/clifford-nvs"
os.makedirs(WORK, exist_ok=True)

import shutil
if REPO_URL:
    if not os.path.isdir(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
else:
    # Copia local del código en Drive: ejecutarlo desde el disco de la sesión es más rápido.
    assert os.path.isdir(f"{CODE_DRIVE}/src"), f"No se encontró el código en {CODE_DRIVE}"
    shutil.rmtree(REPO, ignore_errors=True)
    shutil.copytree(CODE_DRIVE, REPO, ignore=shutil.ignore_patterns("__pycache__", "*.pt", "*.npz"))
%cd {REPO}
!pip -q install -r requirements.txt

import sys

# Ejecuta un comando mostrando su salida en vivo y detiene la celda si falla.
# Se usa en lugar de `!` para no depender de la expansión de variables de IPython.
def run(*args):
    cmd = [str(a) for a in args]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="")
    if proc.returncode:
        raise RuntimeError(f"El comando terminó con código {proc.returncode}: {' '.join(cmd)}")

PY = [sys.executable, "-u", "-m"]   # -u: salida sin búfer para ver el progreso en vivo

In [ ]:
# 2. Datos: srn_cars.zip → .npz de 64×64 en Drive, copiados al disco local de la sesión
import glob, shutil

DRIVE_DATA, DATA = f"{WORK}/data", "/content/data"
needed = [f"cars_{s}_64.npz" for s in ("train", "val", "test")]

if not all(os.path.exists(f"{DRIVE_DATA}/{f}") for f in needed):
    # Rutas directas primero: recorrer todo Drive puede tardar minutos.
    zips = [p for p in ("/content/drive/MyDrive/srn_cars.zip", f"{WORK}/srn_cars.zip") if os.path.exists(p)]
    zips = zips or glob.glob("/content/drive/MyDrive/**/srn_cars.zip", recursive=True)
    if zips:
        zip_path = zips[0]
    else:
        import gdown
        listing = gdown.download_folder(id="1PsT3uKwqHHD2bEEHkIXB99AlIjtmrEiR", skip_download=True, quiet=True)
        file_id = next(f.id for f in listing if f.path.endswith("srn_cars.zip"))
        zip_path = "/content/srn_cars.zip"
        gdown.download(id=file_id, output=zip_path, quiet=False)
    run("unzip", "-q", "-n", zip_path, "-d", "/content/srn")
    run(*PY, "src.data", "--srn_root", "/content/srn", "--out", DRIVE_DATA)

# Leer desde el disco local es varias veces más rápido que desde Drive.
os.makedirs(DATA, exist_ok=True)
for f in needed:
    if not os.path.exists(f"{DATA}/{f}"):
        shutil.copy(f"{DRIVE_DATA}/{f}", f"{DATA}/{f}")
print("Datos listos:", sorted(os.listdir(DATA)))

In [ ]:
# 3. Experimentos
CONFIGS = ["configs/A_conv_matrix.yaml", "configs/B_clifford_matrix.yaml", "configs/C_clifford_rotor.yaml"]
SEEDS = [0, 1]
SESSION_HOURS = 3.5        # tiempo de GPU que se dedicará en esta sesión
# CONFIGS.append("configs/D_conv_rotor.yaml")   # diseño factorial completo, si hay tiempo

import sys
sys.path.insert(0, REPO)
from src.utils import load_config
from src.models import build_model, count_params
for c in CONFIGS:
    cfg = load_config(c)
    print(f"{cfg['name']:20s} {count_params(build_model(cfg['model'])) / 1e6:.3f} M parámetros, "
          f"{cfg['train']['steps']} pasos")

In [ ]:
# 4. Entrenamiento (re-ejecutable: salta lo terminado y reanuda lo interrumpido)
import time
t0 = time.time()
for seed in SEEDS:                      # primero todas las configuraciones con la semilla 0
    for c in CONFIGS:
        left = SESSION_HOURS - (time.time() - t0) / 3600
        if left < 0.05:
            break
        run(*PY, "src.train", "--config", c, "--seed", seed, "--data", DATA, "--work", WORK,
            "--time_budget_h", f"{left:.3f}")
print(f"Tiempo de la sesión: {(time.time() - t0) / 3600:.2f} h")

In [ ]:
# 5. Estado de los runs
import pandas as pd
from src.utils import read_json
rows = []
for c in CONFIGS:
    name = load_config(c)["name"]
    for seed in SEEDS:
        d = f"{WORK}/runs/{name}/seed{seed}"
        log = read_json(f"{d}/log.json") if os.path.exists(f"{d}/log.json") else None
        summ = read_json(f"{d}/summary.json") if os.path.exists(f"{d}/summary.json") else None
        rows.append({"run": name, "seed": seed,
                     "estado": "completo" if summ else ("en curso" if log else "pendiente"),
                     "paso": log["train"][-1]["step"] if log and log["train"] else 0,
                     "mejor val PSNR": max((v["psnr"] for v in log["val"]), default=None) if log else None,
                     "horas GPU": summ["gpu_hours"] if summ else None})
pd.DataFrame(rows)

In [ ]:
# 6. Evaluación en prueba (704 objetos × 16 vistas) y tablas LaTeX
RESULTS = f"{WORK}/results"
run(*PY, "src.evaluate", "--configs", *CONFIGS, "--seeds", *SEEDS, "--data", DATA, "--work", WORK, "--out", RESULTS)

In [ ]:
# 7. Figuras, análisis complementario y copia de registros
FOCUS = "configs/C_clifford_rotor.yaml"   # modelo del análisis de peores casos
run(*PY, "src.figures", "--configs", *CONFIGS, "--seeds", *SEEDS, "--data", DATA, "--work", WORK,
    "--out", RESULTS, "--focus", FOCUS)
run(*PY, "src.analysis", "--configs", *CONFIGS, "--seeds", *SEEDS, "--work", WORK, "--out", RESULTS)

os.makedirs(f"{RESULTS}/logs", exist_ok=True)
for c in CONFIGS:
    name = load_config(c)["name"]
    for seed in SEEDS:
        for f in ("log.json", "summary.json"):
            p = f"{WORK}/runs/{name}/seed{seed}/{f}"
            if os.path.exists(p):
                shutil.copy(p, f"{RESULTS}/logs/{name}_seed{seed}_{f}")

from IPython.display import Image, display
for fig in ("fig_curvas", "fig_psnr_angulo", "fig_cualitativa", "fig_error", "fig_peores_casos"):
    if os.path.exists(f"{RESULTS}/{fig}.png"):
        display(Image(f"{RESULTS}/{fig}.png", width=700))

In [ ]:
# 8. Pesos publicables y paquete de resultados para el repositorio
run(*PY, "src.export", "--configs", *CONFIGS, "--seeds", 0, "--data", DATA, "--work", WORK, "--out", f"{WORK}/release")
shutil.make_archive("/content/results", "zip", RESULTS)
print("1) Comparte la carpeta MyDrive/clifford_nvs/release como 'Cualquier persona con el enlace' y")
print("   copia su ID en README.md y en notebooks/02_demo_inferencia.ipynb.")
print("2) Descarga /content/results.zip y descomprímelo en results/ del repositorio antes del commit.")
from google.colab import files
files.download("/content/results.zip")